# Lab: Can an AI weather model forecast your event?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ai_weather_lab/neuralgcm_event_lab.ipynb)

Pick a weather event, watch it in ERA5, and find out whether NeuralGCM could have forecast it. NeuralGCM (Kochkov et al. 2024) combines a conventional dynamical core with neural networks trained on ERA5.

Before you start, choose `Runtime → Change runtime type → T4 GPU`, then run the four setup cells.

In [ ]:
#@title Setup: install packages (~1 min)
!pip install -q neuralgcm gcsfs cartopy
# If the next cell fails with a version error, uncomment this line, run it, then
# restart the session (Runtime -> Restart session) and continue from the next cell:
# !pip install -q "neuralgcm==1.2.2" "dinosaur==1.3.6"

In [ ]:
#@title Setup: imports and ERA5 data
import pickle
import time

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gcsfs
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils
from IPython.display import HTML, display
from matplotlib import animation, colors, ticker

import neuralgcm

print("JAX devices:", jax.devices())
if jax.default_backend() != "gpu":
    print("\n*** No GPU found. Use Runtime -> Change runtime type -> GPU, then rerun. ***")

plt.rcParams.update({"font.size": 11, "figure.dpi": 100, "animation.embed_limit": 100})
G = 9.80665   # gravity, m/s^2
RD = 287.05   # gas constant of dry air, J/(kg K)

# ERA5 reanalysis, hourly at 0.25 degrees (Google's public copy, no account needed)
ERA5 = xr.open_zarr("gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
                    chunks=None, storage_options=dict(token="anon"))


def pretty_units(units):
    return units.replace("**-1", "⁻¹").replace("**-2", "⁻²").replace("**2", "²")


def show_era5_variables(search=""):
    # A table of the ERA5 variables whose name or description contains `search`,
    # with the ones NeuralGCM can forecast first
    forecastable = set(model.input_variables) | set(SINGLE_LEVEL)
    rows = []
    for name in ERA5.data_vars:
        description = ERA5[name].attrs.get("long_name", "")
        if search.lower() in (name + " " + description).lower():
            rows.append({
                "variable": name,
                "description": description,
                "units": pretty_units(ERA5[name].attrs.get("units", "")),
                "pressure levels": "yes" if "level" in ERA5[name].dims else "",
                "NeuralGCM forecasts it": "yes" if name in forecastable else "",
            })
    table = pd.DataFrame(rows).sort_values(["NeuralGCM forecasts it", "variable"], ascending=[False, True])
    try:   # in Colab: a table you can sort and search
        from google.colab import data_table
        display(data_table.DataTable(table, include_index=False, num_rows_per_page=25))
    except ImportError:
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(table.set_index("variable"))


def crop(ds, lats, lons):
    # Cut out a box: lats = (south, north), lons = (west, east), with negative longitudes to the west.
    # Longitudes in the result run continuously from west to east, even across 0 or 180 degrees.
    west, east = lons
    offset = (ds.longitude - west) % 360
    ds = ds.isel(longitude=np.flatnonzero(offset.values <= (east - west) % 360))
    ds = ds.assign_coords(longitude=west + (ds.longitude - west) % 360).sortby("longitude")
    inside = (ds.latitude.values >= lats[0]) & (ds.latitude.values <= lats[1])
    return ds.isel(latitude=np.flatnonzero(inside)).sortby("latitude")


def in_plot_units(field, variable, level):
    # Convert ERA5 units to friendlier ones, and make a label
    units = field.attrs.get("units", "")
    name = field.attrs.get("long_name", variable.replace("_", " "))
    if variable == "total_precipitation":
        field, units = field * 1000, "mm/h"   # m of water in the past hour -> mm/h
    elif units == "K":
        field, units = field - 273.15, "°C"
    elif units == "Pa":
        field, units = field / 100, "hPa"
    elif units == "m**2 s**-2":
        field, units, name = field / G, "m", name.replace("Geopotential", "Geopotential height")
    elif units == "kg kg**-1":
        field, units = field * 1000, "g/kg"
    label = f"{name} ({pretty_units(units)})"
    if level is not None:
        label = f"{level} hPa {label}"
    field = field.drop_vars("level", errors="ignore")
    return field.assign_coords(time=field.time.values.astype("datetime64[ns]")), label


def event_data(colors, lines):
    # What the maps need: `colors` and `lines` are (field, variable, level), with field still in ERA5 units
    colors_field, colors_label = in_plot_units(*colors)
    lines_field, lines_label = in_plot_units(*lines)
    data = xr.Dataset({"colors": colors_field, "lines": lines_field})
    return data.assign_attrs(colors=colors[1], colors_label=colors_label, lines=lines[1], lines_label=lines_label)


def check_variable(variable, level):
    if variable not in ERA5:
        raise ValueError(f"ERA5 has no variable {variable!r}. Run show_era5_variables() to see the names.")
    if "level" in ERA5[variable].dims and level is None:
        raise ValueError(f"{variable} is on pressure levels: give it a level, for example 500")


def load_era5(variable, level=None, lines="mean_sea_level_pressure", lines_level=None):
    # ERA5 in the box (LATS, LONS), every 6 hours from START to END: `variable` for the colors and `lines` for the
    # contour lines, each at a pressure level if it has levels
    times = np.arange(np.datetime64(START), np.datetime64(END) + np.timedelta64(1, "h"), np.timedelta64(6, "h"))
    tic = time.time()
    loaded = []
    for name, lev in [(variable, level), (lines, lines_level)]:
        check_variable(name, lev)
        field = ERA5[name].sel(time=times)
        if "level" in field.dims:
            print(f"Loading {name} at {lev} hPa; variables on pressure levels take a minute or two...")
            field = field.sel(level=lev)
        loaded.append((crop(field, LATS, LONS).compute(), name, lev))
    print(f"Loaded ERA5 in {time.time() - tic:.0f} s")
    return event_data(*loaded)

In [ ]:
#@title Setup: maps and animations
PC = ccrs.PlateCarree()


def when(t):
    return pd.Timestamp(t).strftime("%b %d %Y, %H UTC")


def box_projection():
    # Map projection centered on the box, and the box's extent in it
    width = (LONS[1] - LONS[0]) % 360
    center = LONS[0] + width / 2
    return ccrs.PlateCarree(central_longitude=center), [-width / 2, width / 2, LATS[0], LATS[1]]


def color_style(data):
    # Color map and color scale for the colors field, from its values over all times
    variable, field = data.attrs["colors"], data.colors
    if variable == "total_precipitation":
        return "turbo", colors.LogNorm(0.1, 30)
    lo, hi = np.nanpercentile(field, [1, 99])
    if "cloud" in variable:   # clouds look like a satellite image: white on black
        return "gray", colors.PowerNorm(0.5, vmin=0, vmax=hi)
    if lo < 0 < hi:   # values of both signs: red above zero, blue below
        top = max(-lo, hi)
        return "RdBu_r", colors.Normalize(-top, top)
    return "viridis", colors.Normalize(lo, hi)


def line_levels(data):
    # Contour levels for the lines field, the same at all times: every 4 hPa for sea-level pressure,
    # otherwise about 12 round numbers across its range
    if data.attrs["lines"] == "mean_sea_level_pressure":
        return np.arange(900, 1080, 4)
    lo, hi = np.nanpercentile(data.lines, [1, 99])
    return ticker.MaxNLocator(12).tick_values(lo, hi)


def lowest_pressure(pressure):
    # The lowest sea-level pressure on a map, with its latitude and longitude
    p = pressure.stack(point=("latitude", "longitude"))
    return p.isel(point=p.argmin("point"))


def draw_map(ax, data, cmap, norm, levels):
    # One map at one time: one field in color, another as labeled lines (and the lowest sea-level pressure)
    proj, extent = box_projection()
    dark = cmap == "gray"
    field = data.colors
    if data.attrs["colors"] == "total_precipitation":
        field = field.where(field >= 0.1)   # leave dry areas blank
        ax.add_feature(cfeature.OCEAN, color="#eef3f7")
        ax.add_feature(cfeature.LAND, color="#e3e3e3")
    ax.pcolormesh(field.longitude, field.latitude, field.transpose("latitude", "longitude"),
                  transform=PC, cmap=cmap, norm=norm, shading="auto")
    line_color = "white" if dark else "black"
    lines = data.lines.transpose("latitude", "longitude")
    contours = ax.contour(lines.longitude, lines.latitude, lines, levels=levels, colors=line_color,
                          linewidths=0.7, transform=PC)
    if data.attrs["lines"] == "mean_sea_level_pressure":   # mark the lowest pressure
        low = lowest_pressure(data.lines)
        ax.text(float(low.longitude), float(low.latitude), f"L {float(low):.0f}", transform=PC, color=line_color,
                ha="center", va="center", fontsize=11,
                bbox=dict(boxstyle="round", fc="black" if dark else "white", ec="none", alpha=0.6))
    else:
        ax.clabel(contours, fontsize=8, fmt="%g")
    ax.coastlines(color="gold" if dark else "0.2", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, edgecolor="gold" if dark else "0.3", linewidth=0.5)
    ax.add_feature(cfeature.STATES, edgecolor="gold" if dark else "0.5", linewidth=0.3)
    ax.set_extent(extent, crs=proj)


def shared_times(runs):
    times = None
    for data in runs.values():
        t = pd.DatetimeIndex(data.time.values).as_unit("ns")
        times = t if times is None else times.intersection(t)
    return times


def animate(runs, save=None):
    # Animated maps, one panel per dataset in `runs` ({name: data}), at the times they share. Color scale and
    # contour levels are set by the first dataset. save = "name.gif" also saves the animation (find it in the Files panel on the left).
    runs = {name: crop(data, LATS, LONS) for name, data in runs.items()}
    times = shared_times(runs)
    if len(times) == 0:
        print("These datasets share no times:")
        for name, data in runs.items():
            print(f"  {name}: {when(data.time.values[0])} to {when(data.time.values[-1])}")
        return
    first = next(iter(runs.values()))
    cmap, norm = color_style(first)
    levels = line_levels(first)
    proj, extent = box_projection()
    panel_height = 5.5 * (extent[3] - extent[2]) / (extent[1] - extent[0])
    fig, axes = plt.subplots(1, len(runs), figsize=(5.5 * len(runs), panel_height + 1.5),
                             subplot_kw={"projection": proj}, layout="constrained", squeeze=False)
    axes = axes[0]
    fig.colorbar(plt.cm.ScalarMappable(norm, cmap), ax=axes, orientation="horizontal", shrink=0.6,
                 label=first.attrs["colors_label"], format="%g")

    def frame(i):
        for ax, (name, data) in zip(axes, runs.items()):
            ax.clear()
            draw_map(ax, data.sel(time=times[i]), cmap, norm, levels)
            ax.set_title(name)
        fig.suptitle(f"{EVENT}, {when(times[i])}\nlines: {first.attrs['lines_label']}")

    print(f"Building the animation ({len(times)} frames)...")
    frame(0)
    anim = animation.FuncAnimation(fig, frame, frames=len(times), interval=400)
    if save:
        anim.save(save, writer="pillow", fps=3)
        print(f"Saved {save}")
    display(HTML(anim.to_jshtml(default_mode="once")))
    plt.close(fig)

In [ ]:
#@title Setup: load NeuralGCM (~1 min)
MODEL_NAME = "v1_precip/stochastic_precip_2_8_deg.pkl"   # the 2.8 degree model that also predicts rain
OUTPUT_HOURS = 6                                         # forecasts are saved every 6 hours

gcs = gcsfs.GCSFileSystem(token="anon")
with gcs.open(f"gs://neuralgcm/models/{MODEL_NAME}", "rb") as f:
    model = neuralgcm.PressureLevelModel.from_checkpoint(pickle.load(f))

# For averaging ERA5 onto the model's grid
era5_grid = spherical_harmonic.Grid(
    latitude_nodes=ERA5.sizes["latitude"],
    longitude_nodes=ERA5.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(ERA5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(ERA5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(era5_grid, model.data_coords.horizontal, skipna=True)


def to_model_grid(ds):
    return xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(ds, regridder))


surface = to_model_grid(ERA5[["geopotential_at_surface"]].sel(time=["2000-01-01T00"]).compute())
surface_geopotential = surface.geopotential_at_surface.isel(time=0, drop=True)


def load_initial_state(start):
    # The ERA5 state at `start`, on the model grid
    raw = (
        ERA5[model.input_variables + model.forcing_variables]
        # The model expects sea surface temperature and sea ice from 24 h earlier, as in training
        .pipe(xarray_utils.selective_temporal_shift,
              variables=model.forcing_variables, time_shift="24 hours")
        .sel(time=[np.datetime64(start)])
        .compute()
    )
    return to_model_grid(raw)


def add_warm_blob(state, lat, lon, kelvin, radius_km=500):
    # Warm the air between 850 and 300 hPa by `kelvin` at (lat, lon), fading out over radius_km
    lat_r, lon_r = np.deg2rad(state.latitude), np.deg2rad(state.longitude)
    cos_distance = (np.sin(lat_r) * np.sin(np.deg2rad(lat))
                    + np.cos(lat_r) * np.cos(np.deg2rad(lat)) * np.cos(lon_r - np.deg2rad(lon)))
    distance_km = 6371 * np.arccos(np.clip(cos_distance, -1, 1))
    blob = kelvin * np.exp(-(distance_km / radius_km) ** 2)
    in_layer = (state.level >= 300) & (state.level <= 850)
    warm = state.copy()
    warm["temperature"] = (state.temperature + blob.where(in_layer, 0.0)).transpose(*state.temperature.dims)
    return warm


def forecast(start, end, nudge=None):
    # Run NeuralGCM from the ERA5 state at `start` until `end`, saving every 6 hours.
    # nudge = (lat, lon, kelvin) adds a warm blob to the starting state first.
    tic = time.time()
    state = load_initial_state(start)
    if nudge is not None:
        state = add_warm_blob(state, *nudge)
    print(f"Loaded the ERA5 starting state in {time.time() - tic:.0f} s; running...")
    inputs = model.inputs_from_xarray(state.isel(time=0))
    forcings = model.forcings_from_xarray(state.isel(time=0))
    model_state = model.encode(inputs, forcings, jax.random.key(0))
    days = int(np.ceil((np.datetime64(end) - np.datetime64(start)) / np.timedelta64(1, "D")))
    pieces = []
    for day in range(days + 1):   # one day at a time, so JAX compiles the model only once
        model_state, out = model.unroll(
            model_state,
            model.forcings_from_xarray(state.head(time=1)),   # sea surface temperature and sea ice stay fixed
            steps=24 // OUTPUT_HOURS,
            timedelta=np.timedelta64(OUTPUT_HOURS, "h"),
            start_with_input=True,
        )
        pieces.append(jax.device_get(out))
    out = jax.tree.map(lambda *days: np.concatenate(days), *pieces)
    hours = np.arange(len(out["temperature"])) * OUTPUT_HOURS
    out = model.data_to_xarray(out, times=hours)
    out = out.assign_coords(time=np.datetime64(start, "ns") + hours * np.timedelta64(1, "h"))
    print(f"Forecast from {when(start)} done in {time.time() - tic:.0f} s")
    return out.sel(time=slice(None, np.datetime64(end, "ns")))


# ---- ERA5 variables computed from the model's pressure levels ----
def above_ground(out):
    # Same grid; copying the coordinates avoids tiny rounding differences between them
    surface = surface_geopotential.assign_coords(latitude=out.latitude, longitude=out.longitude)
    return out.geopotential > surface


def column(out, x):
    # Amount of x in the column above the ground, kg/m^2
    return x.where(above_ground(out), 0.0).integrate("level") * 100 / G


def near_ground(out, x):
    # x on the lowest pressure level that is above the ground
    return x.where(above_ground(out)).sortby("level", ascending=False).bfill("level").isel(level=0)


def sea_level_pressure(out):
    # Pa, from the 1000 hPa height and temperature (hypsometric equation)
    return 100000 * np.exp(out.geopotential.sel(level=1000) / (RD * out.temperature.sel(level=1000)))


def precipitation(out):
    # m of water per hour, averaged over the past 6 hours (ERA5 gives it for the past hour)
    total = out.precipitation_cumulative_mean.squeeze("surface", drop=True)   # m since the start
    return (total.diff("time") / OUTPUT_HOURS).reindex(time=out.time)


SINGLE_LEVEL = {
    "total_precipitation": precipitation,
    "mean_sea_level_pressure": sea_level_pressure,
    "total_column_water_vapour": lambda out: column(out, out.specific_humidity),
    "total_column_cloud_liquid_water": lambda out: column(out, out.specific_cloud_liquid_water_content),
    "total_column_cloud_ice_water": lambda out: column(out, out.specific_cloud_ice_water_content),
    "2m_temperature": lambda out: near_ground(out, out.temperature),
    "10m_u_component_of_wind": lambda out: near_ground(out, out.u_component_of_wind),
    "10m_v_component_of_wind": lambda out: near_ground(out, out.v_component_of_wind),
}


def model_variable(out, variable, level):
    # One ERA5 variable from the forecast, in ERA5's units
    if variable in SINGLE_LEVEL:
        field = SINGLE_LEVEL[variable](out)
    elif variable in out and "level" in out[variable].dims:
        if level is None:
            raise ValueError(f"{variable} is on pressure levels: give it a level, for example 500")
        field = out[variable].sel(level=level)
    else:
        on_levels = [name for name in out.data_vars if "level" in out[name].dims]
        raise ValueError(f"NeuralGCM does not predict {variable}. It predicts these on pressure levels: "
                         f"{', '.join(on_levels)}; and these single-level ones: {', '.join(SINGLE_LEVEL)}")
    return field.assign_attrs(ERA5[variable].attrs)


def model_field(out, variable, level=None, lines="mean_sea_level_pressure", lines_level=None):
    # The forecast, ready to animate next to load_era5 with the same arguments
    return event_data((model_variable(out, variable, level), variable, level),
                      (model_variable(out, lines, lines_level), lines, lines_level))


def show_spread(nudged, control, nudge, save=None):
    # Animated world map of where the nudged forecast's sea-level pressure differs from the control's
    diff = (sea_level_pressure(nudged) - sea_level_pressure(control)) / 100
    diff = diff.transpose("time", "latitude", "longitude")
    lat0, lon0, kelvin = nudge
    norm = colors.SymLogNorm(linthresh=0.1, vmin=-10, vmax=10)
    fig, ax = plt.subplots(figsize=(10, 5.5), subplot_kw={"projection": ccrs.Robinson(np.mean(LONS))},
                           layout="constrained")
    fig.colorbar(plt.cm.ScalarMappable(norm, "RdBu_r"), ax=ax, orientation="horizontal", shrink=0.6,
                 label="nudged - control sea-level pressure (hPa)", ticks=[-10, -1, -0.1, 0, 0.1, 1, 10],
                 format="%g")

    def frame(i):
        ax.clear()
        d = diff.isel(time=i)
        ax.pcolormesh(d.longitude, d.latitude, d, transform=PC, cmap="RdBu_r", norm=norm, shading="auto")
        ax.coastlines(color="0.3", linewidth=0.6)
        west, east = LONS[0], LONS[0] + (LONS[1] - LONS[0]) % 360
        ax.plot([west, east, east, west, west], [LATS[0], LATS[0], LATS[1], LATS[1], LATS[0]],
                color="black", lw=2, transform=PC)
        ax.plot(lon0, lat0, "*", color="gold", ms=18, mec="black", transform=PC)
        ax.set_global()
        ax.set_title(f"{when(d.time.values)}  (star: {kelvin:g} K warm blob; box: {EVENT})")

    print(f"Building the animation ({diff.sizes['time']} frames)...")
    frame(0)
    anim = animation.FuncAnimation(fig, frame, frames=diff.sizes["time"], interval=400)
    if save:
        anim.save(save, writer="pillow", fps=3)
        print(f"Saved {save}")
    display(HTML(anim.to_jshtml(default_mode="once")))
    plt.close(fig)


print(f"Loaded {MODEL_NAME}")

---
## 1. Your event

Type in your event's name, a time window (UTC, on 00, 06, 12 or 18 h) and a box around it. Longitudes west of Greenwich are negative. ERA5 goes back to 1940.

In [ ]:
EVENT = "Hurricane Sandy"
START = "2012-10-25T00"
END = "2012-10-31T00"
LATS = (20, 50)     # south, north
LONS = (-85, -55)   # west, east

# Other events to try:
# Typhoon Tip:                    START = "1979-10-08T00", END = "1979-10-16T00", LATS = (5, 40),  LONS = (120, 160)
# 1993 Storm of the Century:      START = "1993-03-11T00", END = "1993-03-15T00", LATS = (20, 55), LONS = (-100, -60)
# Pacific Northwest heat dome:    START = "2021-06-24T00", END = "2021-07-02T00", LATS = (35, 60), LONS = (-135, -105)
# California atmospheric rivers:  START = "2023-01-01T00", END = "2023-01-11T00", LATS = (25, 50), LONS = (-150, -110)

Now pick what to look at: any ERA5 variable from the table below. For a variable on pressure levels, such as `temperature` or `geopotential`, also pick a level. To compare with NeuralGCM later, pick one it forecasts. The lines on each map are sea-level pressure, every 4 hPa, with the lowest pressure marked L.

In [ ]:
show_era5_variables()

In [ ]:
VARIABLE = "total_column_water_vapour"
LEVEL = None   # pressure level in hPa, only for variables on pressure levels, e.g. 500

era5 = load_era5(VARIABLE, LEVEL)
animate({"ERA5": era5})

---
## 2. Forecast it

Start NeuralGCM from the ERA5 state a few days before your event. The model predicts temperature, geopotential, wind, humidity and cloud water on pressure levels, and rain. From these we also get sea-level pressure, column water vapor and cloud water, and approximate 2 m temperature and 10 m wind.

In [ ]:
FORECAST_START = "2012-10-24T00"
fc = forecast(FORECAST_START, END)

In [ ]:
animate({"ERA5": era5, "NeuralGCM": model_field(fc, VARIABLE, LEVEL)})

---
## 3. Make it yours

Design your own map: one variable in color and another as lines, each with a pressure level if it has levels. Try other start dates or a different event too. When you have the comparison that tells your event's story best, save it and share it with the class. The file appears in the Files panel on the left.

In [ ]:
COLORS, COLORS_LEVEL = "total_precipitation", None
LINES, LINES_LEVEL = "geopotential", 500

mine = {
    "ERA5": load_era5(COLORS, COLORS_LEVEL, LINES, LINES_LEVEL),
    "NeuralGCM": model_field(fc, COLORS, COLORS_LEVEL, LINES, LINES_LEVEL),
}
animate(mine, save="my_event.gif")

---
## Optional: a small change far away

Warm the air by 1 K somewhere far from your event, run the same forecast again, and watch where the two forecasts differ.

In [ ]:
NUDGE = (40, 160, 1.0)   # latitude, longitude, kelvin
nudged = forecast(FORECAST_START, END, nudge=NUDGE)
show_spread(nudged, fc, NUDGE)